In [3]:
#@title Gaussian integrals (copied from gaussian_basis.ipynb)
import numpy as np
import scipy.linalg as slin
from scipy.special import hyp1f1, factorial2

def E(i,j,t,Qx,a,b):
    # Recursive definition of the overlap integral coefficients in terms
    # of Hermite Gaussians. i is the angular quantum number for
    # Gaussian "A", and likewise j is for Gaussian "B".
    # a is the orbital exponent for "A", b for "B".
    # t is the number of nodes in the Hermite polynomial.
    # Qx is the distance between the two Gaussians.
    p = a + b
    q = a * b / p
    if (t < 0) or (t > (i + j)):
        return 0.0
    elif i == j == t == 0:
        return np.exp(-q * Qx * Qx)
    elif j == 0:
        # decrement i and recurse
        return (1/(2*p))*E(i-1,j,t-1,Qx,a,b) - (q*Qx/a)*E(i-1,j,t,Qx,a,b) + (t+1)*E(i-1,j,t+1,Qx,a,b)
    else:
        # decrement j and recurse
        return (1/(2*p))*E(i,j-1,t-1,Qx,a,b) + (q*Qx/b)*E(i,j-1,t,Qx,a,b) + (t+1)*E(i,j-1,t+1,Qx,a,b)


def overlap(a,ijk1,A,b,ijk2,B):
    # Evaluates overlap integral between two Gaussians
    # a is the orbital exponent on Gaussian 'A' (alpha in the notes)
    # b is the orbital exponent on Gaussian 'B' (beta in the notes)
    # ijk1 is a list or tuple containing orbital angular momentum  quantum
    # numbers for Gaussian 'A', e.g. (i,j,k)
    # ijk2: same for Gaussian 'B'
    # A is a list containing the [x,y,z] coordinates of the origin of
    # Gaussian 'A', e.g. [1.0, 2.0, 0.0]
    # B is the same for Gaussian 'B'

    i1, j1, k1 = ijk1
    i2, j2, k2 = ijk2
    S1 = E(i1,i2,0,A[0]-B[0],a,b) # X component of 3D gaussian
    S2 = E(j1,j2,0,A[1]-B[1],a,b) # Y component of 3D gaussian
    S3 = E(k1,k2,0,A[2]-B[2],a,b) # Z component of 3D gaussian
    return S1*S2*S3*np.power(np.pi/(a+b),1.5)


def kinetic(a,ijk1,A,b,ijk2,B):
    # kinetic energy integral between two 3D Gaussians
    # a is the exponent for Gaussian "A", and likewise for b.
    # ijk1 is a list or tuple containing angular momentum quantum
    # numbers for Gaussian "A." ijk2 likewise for Gaussian "B."
    # A is list containing origin in (x,y,z) coordinates of "A". likewise B.
    i1, j1, k1 = ijk1
    i2, j2, k2 = ijk2
    term0 = b*(2*(i2+j2+k2)+3)*\
                            overlap(a,(i1,j1,k1),A,b,(i2,j2,k2),B)
    term1 = -2* b**2 *\
                           (overlap(a,(i1,j1,k1),A,b,(i2+2,j2,k2),B) +
                            overlap(a,(i1,j1,k1),A,b,(i2,j2+2,k2),B) +
                            overlap(a,(i1,j1,k1),A,b,(i2,j2,k2+2),B))
    term2 = -0.5*(i2*(i2-1)*overlap(a,(i1,j1,k1),A,b,(i2-2,j2,k2),B) +
                  j2*(j2-1)*overlap(a,(i1,j1,k1),A,b,(i2,j2-2,k2),B) +
                  k2*(k2-1)*overlap(a,(i1,j1,k1),A,b,(i2,j2,k2-2),B))
    return term0+term1+term2


def R(t,u,v,n,p,PCx,PCy,PCz):
    # Returns the Hermite Coulomb integrals "R" from the notes
    # t,u,v are the order of Coulomb Hermite derivative in x,y,z.
    # n is the order of Boys function.
    # p is the sum of the Gaussian exponents
    # PCx,y,z: components of the vector |P-C|: the vector between Gaussian
    # composite center P and nuclear center C.

    RPC = np.sqrt(PCx**2 + PCy**2 + PCz**2)
    T = p*RPC*RPC
    val = 0.0
    if t == u == v == 0:  # end the recursion here
        val += np.power(-2*p,n)*boys(n,T)
    elif t == u == 0:
        # decrement v and recurse
        if v > 1:
            val += (v-1)*R(t,u,v-2,n+1,p,PCx,PCy,PCz)
        val += PCz*R(t,u,v-1,n+1,p,PCx,PCy,PCz)
    elif t == 0:
        # decrement u and recurse
        if u > 1:
            val += (u-1)*R(t,u-2,v,n+1,p,PCx,PCy,PCz)
        val += PCy*R(t,u-1,v,n+1,p,PCx,PCy,PCz)
    else:
        # decrement t and recurse
        if t > 1:
            val += (t-1)*R(t-2,u,v,n+1,p,PCx,PCy,PCz)
        val += PCx*R(t-1,u,v,n+1,p,PCx,PCy,PCz)
    return val

def boys(n,T):
    return hyp1f1(n+0.5,n+1.5,-T)/(2.0*n+1.0)

def gaussian_product_center(a,A,b,B):
    return (a*A+b*B)/(a+b)


def nuclear_attraction(a,ijk1,A,b,ijk2,B,C):
    # Evaluates the 1-electron Coulomb integrals that results from the potential
    # creates by the stationary nuclei.
    # a,b are the orbital exponent on Gaussians 'A' and 'B' respectively
    # ijk1, idk2 are lists or tuples containing the 3 orbital angular momentum
    #   quantum numbers for Gaussian 'A' and 'B' respectively
    # A is a list containing [x,y,z] coordinates of the origin of Gaussian 'A'
    # B likewise for Gaussian 'B'
    # C is a list containing [x,y,z] coordinates of the origin of the nucleus

    i1,j1,k1 = ijk1
    i2,j2,k2 = ijk2
    p = a + b
    P = gaussian_product_center(a,A,b,B) # Gaussian composite center

    val = 0.0
    for t in range(i1+i2+1):
        for u in range(j1+j2+1):
            for v in range(k1+k2+1):
                val += E(i1,i2,t,A[0]-B[0],a,b) * \
                       E(j1,j2,u,A[1]-B[1],a,b) * \
                       E(k1,k2,v,A[2]-B[2],a,b) * \
                       R(t,u,v,0,p,P[0]-C[0],P[1]-C[1],P[2]-C[2])
    val *= 2*np.pi/p
    return val


def electron_repulsion(a,ijk1,A,b,ijk2,B,c,ijk3,C,d,ijk4,D):
    # This function evaluates the 4 orbital Coulomb integrals using
    # Hermite polynomial tricks. This was not discussed in our notes on Gaussian
    # basis functions, but it corresponds to evaluating the [ij | kl] integrals
    # we introduced during our Hartree Fock lecture.
    # a,b,c,d are the orbital exponent on Gaussian 'A','B','C','D' respectively
    # ijk1, ijk2, ijk3, ijk4 are lists or tuples containing the 3 orbital angular
    #    momentum quantum numbers for Gaussians 'A','B','C','D' respectively.
    # A,B,C,D are list containing the [x,y,z] coordinates of the origin of
    #    Gaussian 'A', 'B','C','D'

    i1,j1,k1 = ijk1
    i2,j2,k2 = ijk2
    i3,j3,k3 = ijk3
    i4,j4,k4 = ijk4
    p = a+b # composite exponent for P (from Gaussians 'a' and 'b')
    q = c+d # composite exponent for Q (from Gaussians 'c' and 'd')
    alpha = p*q/(p+q)
    P = gaussian_product_center(a,A,b,B) # A and B composite center
    Q = gaussian_product_center(c,C,d,D) # C and D composite center

    val = 0.0
    for t in range(i1+i2+1):
        for u in range(j1+j2+1):
            for v in range(k1+k2+1):
                for tau in range(i3+i4+1):
                    for nu in range(j3+j4+1):
                        for phi in range(k3+k4+1):
                            val += E(i1,i2,t,A[0]-B[0],a,b) * \
                                   E(j1,j2,u,A[1]-B[1],a,b) * \
                                   E(k1,k2,v,A[2]-B[2],a,b) * \
                                   E(i3,i4,tau,C[0]-D[0],c,d) * \
                                   E(j3,j4,nu ,C[1]-D[1],c,d) * \
                                   E(k3,k4,phi,C[2]-D[2],c,d) * \
                                   np.power(-1,tau+nu+phi) * \
                                   R(t+tau,u+nu,v+phi,0,\
                                       alpha,P[0]-Q[0],P[1]-Q[1],P[2]-Q[2])

    val *= 2*np.power(np.pi,2.5)/(p*q*np.sqrt(p+q))
    return val

In [4]:
#@title HF routines

def build_h2_basis_sto3g(R_bond):
    # This function should build the Gaussian basis in a way that is simple, general,
    # and useful for the HF code to use. Specifically, this function will build
    # the basis for H2 using only s-type orbitals and the STO-3G exponents
    #
    # The function should return a list of dictionaries, where each dictionary
    # represents all the relevant parameters for a single primitive Gaussian basis function.

    # STO-3G parameters for Hydrogen atom
    #zeta = 1.24
    exponents = np.array([3.42525091, 0.62391373, 0.16885540])
    norms = (2 * exponents / np.pi)**0.75

    # xyz coordinates for nucleus A and nucleus B
    R_A = np.array([-R_bond/2, 0, 0])
    R_B = np.array([R_bond/2, 0, 0])

    basis = []

    # add 3 s-type (ijk=0,0,0) primitive Gaussians for atom A
    for i in range(3):
        basis_func = {
            'exponent': exponents[i],
            'ijk' : (0,0,0),
            'center' : R_A,
            'norm' : norms[i]
        }
        basis.append(basis_func)

    # add 3 s-type (ijk=0,0,0) primitive Gaussians for atom B
    for i in range(3):
        basis_func = {
            'exponent': exponents[i],
            'ijk' : (0,0,0),
            'center' : R_B,
            'norm' : norms[i]
        }
        basis.append(basis_func)

    # we should also return a simple representation of the nuclei to compute
    # the nuclear attraction integrals and the constant nuclear repulsion term.
    nuclei = [
        {'Z': 1.0, 'center': R_A},
        {'Z': 1.0, 'center': R_B}
    ]

    return basis, nuclei


def run_hartree_fock(basis, nuclei, n_elec, max_iter=50, tol=1e-6):
    # runs a restricted Hartree-Fock (RHF) calculation for a closed-shell system.
    # i.e. total number of electrons should be even!

    num_basis = len(basis)
    N_nuc = len(nuclei)
    if (n_elec // 2 - n_elec/2) > 1e-8:
      raise ValueError("number of electrons must be even for RHF!")
    N_occ = n_elec // 2
    print(f"Starting HF routine with {num_basis} basis functions and {n_elec} electrons...")

    # step 1: calculate all 1-electron and 2-electron integrals so we do not need
    # to recompute them at each iteration of the algorithm

    S = np.zeros((num_basis, num_basis))      # overlap matrix
    T = np.zeros((num_basis, num_basis))      # kinetic energy matrix
    Vext = np.zeros((num_basis, num_basis))   # V_ext, nuclear potential matrix
    ERI = np.zeros((num_basis, num_basis, num_basis, num_basis)) # 4-index electron-electron Coulomb integrals

    # 1-electron integrals
    for i in range(num_basis):
        for j in range(num_basis):
            b1 = basis[i]
            b2 = basis[j]
            prefactor_12 = b1['norm'] * b2['norm']

            S[i, j] = prefactor_12 * overlap(b1['exponent'], b1['ijk'], b1['center'],
                                             b2['exponent'], b2['ijk'], b2['center'])

            T[i, j] = prefactor_12 * kinetic(b1['exponent'], b1['ijk'], b1['center'],
                                             b2['exponent'], b2['ijk'], b2['center'])

            # sum nuclear_attraction integral functions over all nuclei to get
            # total nuclear potential. Remember to multiply by -Z because electrons
            # are attracted to the nuclei!
            for n in range(N_nuc):
              nucleus = nuclei[n]
              integral = nuclear_attraction(b1['exponent'], b1['ijk'], b1['center'],
                                             b2['exponent'], b2['ijk'], b2['center'], nucleus['center'])
              Vext[i,j] += prefactor_12 * integral * -1 * nucleus['Z']


            # 2-electron Integrals [i, j | lambda, sigma]
            for lam in range(num_basis):
                for sig in range(num_basis):
                    b3 = basis[lam]
                    b4 = basis[sig]
                    prefactor_34 = b3['norm'] * b4['norm']

                    # compute [i j | lam sig] using electron_repulstion integral function
                    integral = electron_repulsion(b1['exponent'], b1['ijk'], b1['center'],
                                                  b2['exponent'], b2['ijk'], b2['center'],
                                                  b3['exponent'], b3['ijk'], b3['center'],
                                                  b4['exponent'], b4['ijk'], b4['center'])
                    # add integral to the ERI 4-index array
                    ERI[i, j, lam, sig] = prefactor_12 * prefactor_34 * integral

    # the core Hamiltonian is the sum of kinetic and V_ext operators.
    # these operators do not depend on the chi orbitals, so they stay fixed during
    # all iterations of the algorithm
    H_core = T + Vext


    # step 2: make an initial guess for the chi orbtitals / density matrix

    # we can get a decent initial guess by solving the eigenvalue problem
    # for just the core Hamiltonian (H_core * C = E * S * C), ignoring electron repulsion
    E, C = slin.eigh(H_core, S)

    # build initial density matrix P
    # P_lam,sig = 2 * sum_{b}^{occ} C_{lam,b} * C_{sig,b}
    P = np.zeros((num_basis, num_basis))
    for lam in range(num_basis):
        for sig in range(num_basis):
            for b in range(N_occ):
                P[lam, sig] += 2.0 * C[lam,b] * C[sig, b]


    # step 3: the self-consistent field (SCF) iterations!

    E_old = 0.0

    for iteration in range(max_iter):

        # A. build the Fock matrix (F)
        F = np.zeros((num_basis, num_basis))
        for i in range(num_basis):
            for j in range(num_basis):
                # start with the 1-electron core part
                F[i, j] = T[i,j] + Vext[i,j]

                # add the 2-electron part (coulomb J and exchange K)
                for lam in range(num_basis):
                    for sig in range(num_basis):
                        J = ERI[i, j, lam, sig]
                        K = ERI[i, lam, sig, j] * 0.5

                        # How do we include the necessary sum:
                        # sum_{lam, sig} P_{lam, sig} * ( [i j | lam sig] - 0.5 * [i lam | sig j] )
                        F[i, j] += P[lam, sig] * (J-K)

        # B. calculate total electronic energy
        # E_elec = 0.5 * sum_{i, j} P_ij * (H_core_ij + F_ij)
        E_elec = 0.0
        for i in range(num_basis):
            for j in range(num_basis):
                E_elec += 0.5 * P[i, j] * (H_core[i, j] + F[i, j])

        print(f"Iter {iteration:2d} | Electronic Energy: {E_elec:.6f} Hartree")

        # C. check for convergence
        if abs(E_elec - E_old) < tol:
            print("SCF Converged!")
            break
        E_old = E_elec

        # D. if no convergence, solve the HF / Roothaan equations to get updated MOs
        E, C = slin.eigh(F, S)

        # E. use the new solution C to update the density matrix
        P = np.zeros((num_basis, num_basis))
        for lam in range(num_basis):
            for sig in range(num_basis):
                for b in range(N_occ):
                    P[lam, sig] += 2 * C[lam, b] * C[sig, b]


    # after the iterations are done, add nuclear-nuclear Coulomb interaction
    # constant for total energy
    V_nn = 0.0
    for a in range(N_nuc):
        for b in range(N_nuc):
            if a != b:
              nuc_a = nuclei[a]
              nuc_b = nuclei[b]
              dist = np.linalg.norm(nuc_a['center'] - nuc_b['center'])
              V_nn += 0.5 * nuc_a['Z'] * nuc_b['Z'] / dist



    E_total = E_elec + V_nn

    print("\n--- Final Results ---")
    print(f"Electronic Energy: {E_elec:.6f} Hartree")
    print(f"Nuclear Repulsion: {V_nn:.6f} Hartree")
    print(f"Total HF Energy:   {E_total:.6f} Hartree")

    return E_total


if __name__ == "__main__":
    # Test our H2 Hartree Fock implementation
    bond_length = 1.4 # ~0.74 Angstroms in Bohr (equilibrium for H2)
    nelec = 2
    h2_basis, h2_nuclei = build_h2_basis_sto3g(bond_length)

    run_hartree_fock(h2_basis, h2_nuclei, nelec)

Starting HF routine with 6 basis functions and 2 electrons...
Iter  0 | Electronic Energy: -1.784565 Hartree
Iter  1 | Electronic Energy: -1.832673 Hartree
Iter  2 | Electronic Energy: -1.834253 Hartree
Iter  3 | Electronic Energy: -1.834301 Hartree
Iter  4 | Electronic Energy: -1.834303 Hartree
Iter  5 | Electronic Energy: -1.834303 Hartree
SCF Converged!

--- Final Results ---
Electronic Energy: -1.834303 Hartree
Nuclear Repulsion: 0.714286 Hartree
Total HF Energy:   -1.120017 Hartree
